In [ ]:
import pandas as pd
from pathlib import Path
import pyarrow.parquet as pq
import pyarrow as pa

def merge_parquet_files(input_folder, output_file):
    """
    Merge all Parquet files in a folder into a single Parquet file.
    
    Args:
        input_folder (str): Path to folder containing Parquet files
        output_file (str): Path for output Parquet file
    """
    # Convert to Path objects
    input_path = Path(input_folder)
    output_path = Path(output_file)
    
    # Find all parquet files in the input folder
    parquet_files = list(input_path.glob("*.parquet"))
    
    if not parquet_files:
        print(f"No Parquet files found in {input_folder}")
        return
    
    print(f"Found {len(parquet_files)} Parquet files to merge")
    
    # Method 1: Using pandas (for smaller datasets)
    # Read and concatenate all files
    dfs = []
    for file in parquet_files:
        print(f"Reading {file.name}")
        df = pd.read_parquet(file)
        dfs.append(df)
    
    # Concatenate all DataFrames
    merged_df = pd.concat(dfs, ignore_index=True)
    
    # Save to single Parquet file
    merged_df.to_parquet(output_path, index=False)
    
    print(f"Successfully merged {len(parquet_files)} files")
    print(f"Output: {output_path}")
    print(f"Total rows: {len(merged_df)}")

def merge_parquet_files_efficient(input_folder, output_file):
    """
    More efficient method using PyArrow directly (better for large datasets)
    """
    input_path = Path(input_folder)
    output_path = Path(output_file)
    
    parquet_files = list(input_path.glob("*.parquet"))
    
    if not parquet_files:
        print(f"No Parquet files found in {input_folder}")
        return
    
    print(f"Found {len(parquet_files)} Parquet files to merge")
    
    # Read all files with PyArrow
    tables = []
    for file in parquet_files:
        print(f"Reading {file.name}")
        table = pq.read_table(file)
        tables.append(table)
    
    # Concatenate tables
    merged_table = pa.concat_tables(tables)
    
    # Write to single Parquet file
    pq.write_table(merged_table, output_path)
    
    print(f"Successfully merged {len(parquet_files)} files")
    print(f"Output: {output_path}")
    print(f"Total rows: {merged_table.num_rows}")

if __name__ == "__main__":
    # Example usage
    input_folder = "/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/classification_chunks"  # Replace with your folder path
    output_file = "/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/semi_merged_data.parquet"  # Replace with desired output path
    
    # Use basic method
    merge_parquet_files(input_folder, output_file)
    
    # Or use efficient method for larger datasets
    # merge_parquet_files_efficient(input_folder, output_file)

In [ ]:
# find the failed and missing ones and retry them
semi_merged = pd.read_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/semi_merged_data.parquet")
semi_merged.info()
semi_merged = semi_merged[~(semi_merged['social_encounter'].isin(['FAILED', 'MISSING'])) &
                          ~(semi_merged['racial_connotations'].isin(['FAILED', 'MISSING'])) &
                          ~(semi_merged['connotation_type'].isin(['FAILED', 'MISSING']))]
semi_merged.info()

In [ ]:
semi_merged.connotation_type.value_counts()

In [ ]:
#recovered data
recovered_data = pd.read_csv("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/classification_chunks/recovered_success.csv")
recovered_data.info()
# recovered_data = recovered_data[~(recovered_data['social_encounter'].isin(['FAILED', 'MISSING', 'NOT_PROCESSED'])) &
#                           ~(recovered_data['racial_connotations'].isin(['FAILED', 'MISSING', 'NOT_PROCESSED'])) &
#                           ~(recovered_data['connotation_type'].isin(['FAILED', 'MISSING', 'NOT_PROCESSED']))]
# recovered_data.info()


In [ ]:
## other classiffied data
first_classed = pd.read_csv("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/cs_srt_labeled_26112025.csv")
first_classed.info()
# first_classed = first_classed[~(first_classed['social_encounter'].isin(['FAILED', 'MISSING', 'NOT_PROCESSED'])) &
#                           ~(first_classed['racial_connotations'].isin(['FAILED', 'MISSING', 'NOT_PROCESSED'])) &
#                           ~(first_classed['connotation_type'].isin(['FAILED', 'MISSING', 'NOT_PROCESSED']))]
# first_classed.info()

In [ ]:
# merge all classified data 
merged_classed = pd.concat([semi_merged, recovered_data, first_classed], ignore_index=True).drop_duplicates(subset=['review_id'])

merged_classed.info()

In [ ]:
merged_classed.nunique()

In [ ]:
merged_classed.classification_status.value_counts()

In [ ]:
merged_classed.social_encounter.value_counts()

In [ ]:
merged_classed.to_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/classified_reviews_05122025.parquet", index=False)

In [ ]:
#all reviews
text_reviews = pd.read_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data (copy)/text_reviews_summarized_25112025.parquet")
text_reviews.info()

In [ ]:
unprocessed_reviews = text_reviews[~text_reviews['review_id'].isin(merged_classed['review_id'])]
unprocessed_reviews.info()

In [ ]:
unprocessed_reviews.to_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/reprocessing/unprocessed_rerun.parquet")